In [1]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append(f"./../")

import matplotlib.pyplot as plt
import numpy as np
import random
import os
import networkx as nx

from src.graphs import StaticGraph, DynamicGraph, IntersectingEdgesGraph, MultiEdgeGraph, StaticGraph
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, SparsePauliOp, Pauli
from scipy.linalg import expm
import itertools

from IPython.display import display
import networkx as nx
import random
from collections import defaultdict

In [2]:
def count_edges(G):
    return sum(len(neighbors) for neighbors in G.adj.values()) // 2

def calculate_density(G):
    n = G.number_of_nodes()
    return 2 * G.number_of_edges() / (n * (n-1))

def is_bipartite(G):
    color = {}
    for start_node in G.nodes():
        if start_node not in color:
            stack = [(start_node, 0)]
            while stack:
                node, c = stack.pop()
                if node in color:
                    if color[node] != c:
                        return False
                else:
                    color[node] = c
                    stack.extend((neighbor, 1 - c) for neighbor in G.adj[node])
    return True

def find_diameter(G):
    if not nx.is_connected(G):
        return float('inf')
    def bfs(start):
        distances = {start: 0}
        queue = [start]
        while queue:
            node = queue.pop(0)
            for neighbor in G.adj[node]:
                if neighbor not in distances:
                    distances[neighbor] = distances[node] + 1
                    queue.append(neighbor)
        return max(distances.values())
    
    return max(bfs(node) for node in G.nodes())

def find_max_clique(G):
    def backtrack(candidates, clique):
        if not candidates:
            return clique
        v = max(candidates, key=lambda x: len(G.adj[x]))
        candidates.remove(v)
        new_clique = clique | {v}
        for u in list(candidates):
            if not all(u in G.adj[w] for w in new_clique):
                candidates.remove(u)
        return max((backtrack(candidates.copy(), new_clique), clique), key=len)

    return len(backtrack(set(G.nodes()), set()))

def average_clustering(G):
    def local_clustering(node):
        neighbors = list(G.adj[node])
        if len(neighbors) < 2:
            return 0
        links = sum(1 for u in neighbors for v in neighbors if u < v and v in G.adj[u])
        possible_links = len(neighbors) * (len(neighbors) - 1) / 2
        return links / possible_links if possible_links > 0 else 0
    
    if len(G.nodes()) == 0:
        return 0
    return sum(local_clustering(node) for node in G.nodes()) / len(G.nodes())

def estimate_group_size(G):
    degree_counts = defaultdict(int)
    for node in G.nodes():
        degree_counts[len(G.adj[node])] += 1
    return max(degree_counts.values()) if degree_counts else 0

def estimate_orbit_count(G):
    return len(set(len(G.adj[node]) for node in G.nodes()))

def verify_g6_file(filename):
    """Verify graphs in g6 file meet properties."""
    valid_count = 0
    total_count = 0
    
    with open(filename, 'r') as f:
        for line in f:
            total_count += 1
            try:
                G = nx.from_graph6_bytes(line.strip().encode('ascii'))
                
                # Verify properties
                props = {
                    'density': calculate_density(G),
                    'diameter': find_diameter(G),
                    'clustering': average_clustering(G),
                    'is_bipartite': is_bipartite(G)
                }
                
                # Adjust density check for bipartite graphs
                if (target_props['density_min'] <= props['density'] <= target_props['density_max'] and
                    target_props['diameter_min'] <= props['diameter'] <= target_props['diameter_max'] and
                    target_props['clustering_min'] <= props['clustering'] <= target_props['clustering_max'] and
                    props['is_bipartite']):
                    valid_count += 1
                    
                    if total_count % 10 == 0:  # Print progress every 10 graphs
                        print(f"Verified {total_count} graphs, {valid_count} valid")
                
                del G  # Clear memory
            except Exception as e:
                print(f"Error reading line {total_count}: {e}")
    
    return valid_count, total_count


In [3]:
target_props = {
        'density_max': 0.417,
        'density_min': 0.363,
        'diameter_min': 3.22,
        'diameter_max': 4.5,
        'clustering_min': 0.01,
        'clustering_max': 0.37,
        'is_bipartite' : True
    }

In [4]:
import networkx as nx
import random
from collections import defaultdict
import gc

def generate_synthetic_graph(n_vertices=16, max_attempts=100):
    """Memory-efficient graph generator with specific constraints"""
    target_props = {
        'density_max': 0.678,
        'diameter_min': 3.22,
        'clustering_max': 0.37,
        'max_degree_limit': 4.89,
        'max_clique_limit': 3.16,
        'is_bipartite': True
    }
    
    best_graph = None
    min_score = float('inf')
    
    for attempt in range(max_attempts):
        gc.collect()
        
        G = nx.Graph()
        part1 = list(range(n_vertices//2))
        part2 = list(range(n_vertices//2, n_vertices))
        G.add_nodes_from(range(n_vertices))
        
        # Create initial path to ensure minimum diameter
        path_length = random.randint(4, 6)
        path_start = random.randint(0, len(part1)-path_length)
        
        path_edges = []
        for i in range(path_length-1):
            if i % 2 == 0:
                edge = (part1[path_start + i//2], part2[path_start + i//2])
            else:
                edge = (part2[path_start + i//2], part1[path_start + i//2 + 1])
            path_edges.append(edge)
        
        G.add_edges_from(path_edges)
        
        edge_candidates = []
        for i in part1:
            for j in part2:
                if not G.has_edge(i, j) and (i, j) not in path_edges:
                    edge_candidates.append((i, j))
        
        random.shuffle(edge_candidates)
        
        batch_size = 10
        for idx in range(0, len(edge_candidates), batch_size):
            current_batch = edge_candidates[idx:idx + batch_size]
            current_density = calculate_density(G)
            
            if current_density >= target_props['density_max']:
                break
            
            for u, v in current_batch:
                # Quick degree check before adding edge
                if len(G.adj[u]) >= target_props['max_degree_limit'] or len(G.adj[v]) >= target_props['max_degree_limit']:
                    continue
                    
                G.add_edge(u, v)
                
                # Quick property checks
                max_degree = max(len(G.adj[node]) for node in G.nodes())
                current_density = calculate_density(G)
                
                if (max_degree > target_props['max_degree_limit'] or
                    current_density > target_props['density_max']):
                    G.remove_edge(u, v)
                    continue
                
                try:
                    props = {
                        'density': current_density,
                        'diameter': find_diameter(G),
                        'clustering': average_clustering(G),
                        'is_bipartite': is_bipartite(G),
                        'max_clique': find_max_clique(G),
                        'max_degree': max_degree
                    }
                    
                    # Check if violates any hard constraints
                    if (not props['is_bipartite'] or
                        props['max_clique'] > target_props['max_clique_limit'] or
                        props['density'] > target_props['density_max'] or
                        props['clustering'] > target_props['clustering_max'] or
                        props['diameter'] < target_props['diameter_min']):
                        G.remove_edge(u, v)
                        continue
                    
                    # Calculate score based on how close we are to ideal values
                    score = 0
                    
                    # Penalties for approaching limits
                    score += abs(props['density'] - target_props['density_max'])/target_props['density_max']
                    score += abs(props['diameter'] - target_props['diameter_min'])/target_props['diameter_min']
                    score += props['clustering']/target_props['clustering_max']
                    score += props['max_degree']/target_props['max_degree_limit']
                    score += props['max_clique']/target_props['max_clique_limit']
                    
                    if score < min_score:
                        if best_graph is not None:
                            del best_graph
                        min_score = score
                        best_graph = G.copy()
                    
                except Exception as e:
                    G.remove_edge(u, v)
                    continue
            
            del current_batch
            gc.collect()
        
        # Final check of current graph
        if G is not None:
            try:
                props = {
                    'density': calculate_density(G),
                    'diameter': find_diameter(G),
                    'clustering': average_clustering(G),
                    'is_bipartite': is_bipartite(G),
                    'max_clique': find_max_clique(G),
                    'max_degree': max(len(G.adj[node]) for node in G.nodes())
                }
                
                if (props['density'] <= target_props['density_max'] and
                    props['diameter'] >= target_props['diameter_min'] and
                    props['clustering'] <= target_props['clustering_max'] and
                    props['is_bipartite'] and
                    props['max_clique'] <= target_props['max_clique_limit'] and
                    props['max_degree'] <= target_props['max_degree_limit']):
                    return G
            except Exception:
                pass
    
    return best_graph

def generate_and_save_graphs(output_file, n_graphs=100, n_vertices=16, max_attempts=100):
    """Generate and save graphs with memory management"""
    successful = 0
    
    with open(output_file, 'w') as f:
        while successful < n_graphs:
            try:
                print(f"Generating graph {successful+1}/{n_graphs}")
                G = generate_synthetic_graph(n_vertices, max_attempts)
                
                if G is not None:
                    props = {
                        'density': calculate_density(G),
                        'diameter': find_diameter(G),
                        'clustering': average_clustering(G),
                        'is_bipartite': is_bipartite(G),
                        'max_clique': find_max_clique(G),
                        'max_degree': max(len(G.adj[node]) for node in G.nodes())
                    }
                    
                    print(f"  Properties:")
                    print(f"    Density: {props['density']:.3f} (< 0.678)")
                    print(f"    Diameter: {props['diameter']:.2f} (> 3.22)")
                    print(f"    Clustering: {props['clustering']:.3f} (< 0.37)")
                    print(f"    Is Bipartite: {props['is_bipartite']}")
                    print(f"    Max Clique: {props['max_clique']} (< 3.16)")
                    print(f"    Max Degree: {props['max_degree']} (< 4.89)")
                    
                    # Verify all constraints
                    if (props['density'] <= 0.678 and
                        props['diameter'] >= 3.22 and
                        props['clustering'] <= 0.37 and
                        props['is_bipartite'] and
                        props['max_clique'] <= 3.16 and
                        props['max_degree'] <= 4.89):
                        
                        g6_bytes = nx.to_graph6_bytes(G, header=False)
                        g6_string = g6_bytes.decode('ascii').strip()
                        f.write(f"{g6_string}\n")
                        f.flush()
                        successful += 1
                    else:
                        print("    Failed to meet constraints")
                
                del G
                gc.collect()
                
            except Exception as e:
                print(f"Error: {str(e)}")
                continue
    
    return successful

if __name__ == "__main__":
    output_file = "./../data/graphs/graph16c.g6"
    
    print("\nGenerating 100 synthetic graphs...")
    successful = generate_and_save_graphs(
        output_file=output_file,
        n_graphs=10,
        n_vertices=16,
        max_attempts=100
    )
    
    print(f"\nGenerated {successful} graphs")


Generating 100 synthetic graphs...
Generating graph 1/10
  Properties:
    Density: 0.267 (< 0.678)
    Diameter: 4.00 (> 3.22)
    Clustering: 0.000 (< 0.37)
    Is Bipartite: True
    Max Clique: 2 (< 3.16)
    Max Degree: 4 (< 4.89)
Generating graph 2/10
  Properties:
    Density: 0.258 (< 0.678)
    Diameter: 4.00 (> 3.22)
    Clustering: 0.000 (< 0.37)
    Is Bipartite: True
    Max Clique: 2 (< 3.16)
    Max Degree: 4 (< 4.89)
Generating graph 3/10
  Properties:
    Density: 0.250 (< 0.678)
    Diameter: 4.00 (> 3.22)
    Clustering: 0.000 (< 0.37)
    Is Bipartite: True
    Max Clique: 2 (< 3.16)
    Max Degree: 4 (< 4.89)
Generating graph 4/10
  Properties:
    Density: 0.267 (< 0.678)
    Diameter: 4.00 (> 3.22)
    Clustering: 0.000 (< 0.37)
    Is Bipartite: True
    Max Clique: 2 (< 3.16)
    Max Degree: 4 (< 4.89)
Generating graph 5/10
  Properties:
    Density: 0.267 (< 0.678)
    Diameter: 4.00 (> 3.22)
    Clustering: 0.000 (< 0.37)
    Is Bipartite: True
    Max Cliqu

In [5]:
#output_file = "./../data/graphs/graph16c.g6"